# Errors

> fastllm errors.

In [ ]:
#| default_exp errors

## Imports

In [ ]:
#| export
from fastcore.utils import *
import json, httpx

In [ ]:
#| hide
from cachy.core import enable_cachy
from fastcore.test import *

In [ ]:
enable_cachy()

## Purpose And Design

`errors.py` defines the error contract used across the entire library.

### Why this module exists

Providers return different error shapes for HTTP and streaming failures. If each layer handled raw provider payloads directly, calling code would need provider-specific branches everywhere.

### Architectural fit

- Upstream inputs: raw `httpx` exceptions and SSE error events.
- Downstream consumers: `transport`, dynamic ops in `oapi`, provider `clients`, and high-level entrypoints.

### Key design choices

- `APIError` carries normalized fields (`status_code`, `error_type`, `code`, `request_id`, `retryable`, `raw`).
- Best-effort parsing keeps useful diagnostics without overfitting to one vendor.
- Context enrichment (`provider`, `model`, `endpoint`) helps debugging in multi-provider runs.

### Reader outcome

After this notebook, you should be able to trace where any failure is normalized and how error metadata remains consistent from low-level transport to user-facing calls.

### `FastLLMError`

Base fastllm error.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class FastLLMError(Exception):
    "Base fastllm error."

### `UnsupportedCapabilityError`

Raised when a requested feature is unsupported.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `clients`, `files`, `oapi`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class UnsupportedCapabilityError(FastLLMError):
    "Raised when a requested feature is unsupported."

### `ProtocolError`

Raised when provider payloads do not match expected protocol shape.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `normalize`, `transport`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

What is meant by protocol here in this context?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In this context, "protocol" refers to the expected structure/shape of the data that LLM providers send back — things like the JSON response format, SSE streaming event structure, field names, etc.

So `ProtocolError` gets raised when a provider returns something that doesn't match the shape the library is expecting to parse — e.g. a missing field, an unexpected JSON structure, or a malformed streaming event.

<details class='token-usage-details'><summary>$0.0962</summary>

`total=9,820 | in=9,584 | out=236 | cached=0.0% | cache_new=8,263 | $0.0962`

</details>


do you have access to pyrun and bash tools?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, I have access to both `pyrun` and `bash` tools. `pyrun` runs Python in the same persistent kernel as you, and `bash` runs shell commands through a safe whitelist-checked wrapper.

<details class='token-usage-details'><summary>$0.2462</summary>

`total=26,149 | in=26,099 | out=50 | cached=0.0% | cache_new=25,900 | $0.2462`

</details>


In [ ]:
#| export
class ProtocolError(FastLLMError):
    "Raised when provider payloads do not match expected protocol shape."

### `SpecError`

Raised when an OpenAPI spec cannot be parsed as expected.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `spec`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class SpecError(FastLLMError):
    "Raised when an OpenAPI spec cannot be parsed as expected."

### `_to_text`

Best-effort stringify helper.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_to_text(v: Any)`
- Returns: `str`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _to_text(v):
    "Best-effort stringify helper."
    if v is None: return ""
    if isinstance(v, str): return v
    try: return json.dumps(v, ensure_ascii=False)
    except Exception: return str(v)

### `_retryable`

Classify transient/retryable API failures.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_retryable(status_code: Optional[int], error_type: Any, code: Any, message: str)`
- Returns: `bool`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _retryable(status_code, error_type, code, message):
    "Classify transient/retryable API failures."
    if isinstance(status_code, int) and (status_code >= 500 or status_code in (408, 409, 425, 429)): return True
    t = str(error_type or "").lower()
    c = str(code or "").lower()
    m = (message or "").lower()
    hints = ("server_error", "internal", "overload", "rate_limit", "timeout", "unavailable", "temporar")
    if any(h in t for h in hints): return True
    if any(h in c for h in hints): return True
    if any(h in m for h in ("try again", "server error", "temporar", "timeout", "overloaded", "unavailable")): return True
    return False

Can you please explain these status codes

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! Here's what each one means:

- **500+** — Server errors (500 Internal Server Error, 502 Bad Gateway, 503 Service Unavailable, etc.). The problem is on the provider's end, so retrying makes sense.
- **408** — **Request Timeout**. The server timed out waiting for the request.
- **409** — **Conflict**. The request conflicts with current server state (e.g. concurrent modification). Often transient.
- **425** — **Too Early**. The server is unwilling to process a request that might be replayed. Relatively rare.
- **429** — **Too Many Requests**. Rate limiting — you've hit the provider's request quota. Retrying after a backoff usually works.

The common thread: all of these are *transient* failures where the same request might succeed if retried later, as opposed to something like a 401 (bad auth) or 404 (not found) where retrying won't help.

<details class='token-usage-details'><summary>$0.0770</summary>

`total=27,495 | in=27,218 | out=277 | cached=79.3% | cache_new=4,425 | $0.0770`

</details>


### `_req_id`

Extract provider request id header when available.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_req_id(headers: Optional[httpx.Headers])`
- Returns: `str`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _req_id(headers):
    "Extract provider request id header when available."
    if headers is None: return ""
    for k in ("x-request-id", "request-id", "anthropic-request-id", "x-goog-request-id"):
        v = headers.get(k)
        if v: return str(v)
    return ""

All three major providers (OpenAI, Anthropic, Gemini) consistently return `{"error": <dict>}` with a `message` field inside, so the primary `isinstance(err, dict)` branch handles them all. The remaining branches are defensive fallbacks for edge cases:

- `elif err is not None` — `error` is a string (e.g. proxy/gateway errors)
- `else` — no `error` key at all (e.g. OAuth `error_description`, or `detail`-only responses)
- `except Exception: raw = self.text` — non-JSON responses from CDNs/gateways

Key per-provider differences in the dict branch:
- **[OpenAI](https://platform.openai.com/docs/guides/error-codes)**: `error.{message, type, code, param}`, top-level `status` (int)
- **[Anthropic](https://docs.anthropic.com/en/api/errors)**: `error.{type, message}` only — no `code` or `status`
- **[Gemini](https://ai.google.dev/api/generate-content)**: `error.{code (int), message, status (string), details}` — `status` carries the semantic gRPC code (e.g. `INVALID_ARGUMENT`)

The `status`→`code` promotion at the end ensures semantic codes like Gemini's `"INVALID_ARGUMENT"` are preferred over numeric HTTP codes.

In [ ]:
#| export
@patch
def error(self: httpx.Response):
    "Parse common HTTP API error shapes to (message, error_type, code, raw)."
    try: raw = self.json()
    except Exception: raw = self.text

    msg, et, code, status = "", "", None, None
    if isinstance(raw, dict):
        err = raw.get("error")
        if isinstance(err, dict):
            msg = str(err.get("message") or raw.get("message") or raw.get("detail") or "")
            status = err.get("status", raw.get("status"))
            et = str(err.get("type") or status or raw.get("type") or "")
            code = err.get("code", raw.get("code"))
        elif err is not None:
            msg = _to_text(err)
            status = raw.get("status")
            et = str(raw.get("type") or status or "")
            code = raw.get("code")
        else:
            msg = str(raw.get("message") or raw.get("detail") or raw.get("error_description") or "")
            status = raw.get("status")
            et = str(raw.get("type") or status or "")
            code = raw.get("code")
        if not msg: msg = _to_text(raw)
    else:
        msg = _to_text(raw)
    # Prefer semantic provider codes (e.g. Gemini `status`) over numeric HTTP-like codes.
    if (code is None or isinstance(code, int)) and isinstance(status, str) and status:
        code = status
    if code in (None, "") and isinstance(et, str) and et:
        code = et
    return AttrDict(message=msg, type=et, code=code, raw=raw)

In [ ]:
resp = httpx.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": "Bearer sk-fake123"},
    json={"model": "gpt-4o", "messages": [{"role": "user", "content": "hi"}]}
)
resp.json()

{'error': {'message': 'Incorrect API key provided: sk-fake123. You can find your API key at https://platform.openai.com/account/api-keys.',
  'type': 'invalid_request_error',
  'code': 'invalid_api_key',
  'param': None},
 'status': 401}

In [ ]:
resp.error()

```python
{ 'code': 'invalid_api_key',
  'message': 'Incorrect API key provided: sk-fake123. You can find your API '
             'key at https://platform.openai.com/account/api-keys.',
  'raw': { 'error': { 'code': 'invalid_api_key',
                      'message': 'Incorrect API key provided: sk-fake123. You '
                                 'can find your API key at '
                                 'https://platform.openai.com/account/api-keys.',
                      'param': None,
                      'type': 'invalid_request_error'},
           'status': 401},
  'type': 'invalid_request_error'}
```

In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hi"}]}
)
resp.json()

{'type': 'error',
 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'},
 'request_id': 'req_011CZawgT2EDaKt1J84fwjww'}

In [ ]:
resp.error()

```python
{ 'code': 'authentication_error',
  'message': 'invalid x-api-key',
  'raw': { 'error': { 'message': 'invalid x-api-key',
                      'type': 'authentication_error'},
           'request_id': 'req_011CZawgT2EDaKt1J84fwjww',
           'type': 'error'},
  'type': 'authentication_error'}
```

In [ ]:
resp = httpx.post(
    "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key=fake123",
    json={"contents": [{"parts": [{"text": "hi"}]}]}
)
resp.json()

{'error': {'code': 400,
  'message': 'API key not valid. Please pass a valid API key.',
  'status': 'INVALID_ARGUMENT',
  'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo',
    'reason': 'API_KEY_INVALID',
    'domain': 'googleapis.com',
    'metadata': {'service': 'generativelanguage.googleapis.com'}},
   {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage',
    'locale': 'en-US',
    'message': 'API key not valid. Please pass a valid API key.'}]}}

In [ ]:
resp.error()

```python
{ 'code': 'INVALID_ARGUMENT',
  'message': 'API key not valid. Please pass a valid API key.',
  'raw': { 'error': { 'code': 400,
                      'details': [ { '@type': 'type.googleapis.com/google.rpc.ErrorInfo',
                                     'domain': 'googleapis.com',
                                     'metadata': { 'service': 'generativelanguage.googleapis.com'},
                                     'reason': 'API_KEY_INVALID'},
                                   { '@type': 'type.googleapis.com/google.rpc.LocalizedMessage',
                                     'locale': 'en-US',
                                     'message': 'API key not valid. Please '
                                                'pass a valid API key.'}],
                      'message': 'API key not valid. Please pass a valid API '
                                 'key.',
                      'status': 'INVALID_ARGUMENT'}},
  'type': 'INVALID_ARGUMENT'}
```

> Prefer semantic provider codes (e.g. Gemini `status`) over numeric HTTP-like codes.

I'm trying to understand this function, can you explain this comment

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Look at the Gemini error response above — its `error` dict has both:
- `"code": 400` — a numeric HTTP status code
- `"status": "INVALID_ARGUMENT"` — a human-readable semantic string

The code block right below that comment:

```python
if (code is None or isinstance(code, int)) and isinstance(status, str) and status:
    code = status
```

…says: if `code` is either missing or just a number (like `400`), and there's a descriptive string `status` available (like `"INVALID_ARGUMENT"`), replace `code` with that string. The result is that `resp.error().code` returns `'INVALID_ARGUMENT'` instead of `400` — which is more useful for programmatic error handling downstream, since you already have the HTTP status code in `resp.status_code`.

<details class='token-usage-details'><summary>$0.2951</summary>

`total=30,611 | in=30,145 | out=466 | cached=0.0% | cache_new=27,466 | $0.2951`

</details>


Can you walk me through this function line by line having all 3 provider example error responses, annotate each line why each needed. Let's try to see if it can be simplified

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's trace through with all three responses side by side. I'll go section by section.

**The three raw JSONs for reference:**

| | OpenAI | Anthropic | Gemini |
|---|---|---|---|
| `error` | `{message, type, code, param}` | `{type, message}` | `{code, message, status, details}` |
| top-level extras | `status: 401` | `type: "error"`, `request_id` | *(none)* |

**Lines 1–2: Parse the body**
```python
try: raw = self.json()
except Exception: raw = self.text
```
All three return JSON, so `raw` is the dict. The fallback handles edge cases where a provider returns plain text (e.g. a gateway error page).

**Lines 4–5:**
```python
msg, et, code, status = "", "", None, None
if isinstance(raw, dict):
    err = raw.get("error")
```
All three have `raw["error"]` as a dict, so we enter the `isinstance(err, dict)` branch.

**Now the core extraction — line by line:**

```python
msg = str(err.get("message") or raw.get("message") or raw.get("detail") or "")
```
- **OpenAI**: `err["message"]` → `"Incorrect API key..."`
- **Anthropic**: `err["message"]` → `"invalid x-api-key"`
- **Gemini**: `err["message"]` → `"API key not valid..."`

All three have `message` inside `err`, so the fallbacks to `raw.get("message")` and `raw.get("detail")` don't fire here. Those exist for providers that put the message at the top level or use `detail` (like some OAuth endpoints).

Shall I continue with the `status`/`et`/`code` lines next? That's where it gets interesting with the divergence between providers.

<details class='token-usage-details'><summary>$0.1023</summary>

`total=31,891 | in=30,443 | out=1,448 | cached=90.2% | cache_new=2,728 | $0.1023`

</details>


Use web search to find the official docs about the expected errors from these providers, and see if all the code here is needed and if we can assume things

Don't use parallel web search it causes this error

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Got it, I'll search sequentially. Let me look up each provider's error format one at a time.



OK, here's what I found about each provider's documented error shapes:

**OpenAI** — always `{"error": {"message": "...", "type": "...", "code": "...", "param": ...}}`. [*](https://community.openai.com/t/openai-chat-list-of-error-codes-and-types/357791 "OpenAI chat list of error codes and types - API - OpenAI Developer Community") The response has fields like `type`, `code` inside the `error` dict. Sometimes a top-level `status` (numeric HTTP code) is included too, as we saw in the example.

**Anthropic** — [*](https://docs.anthropic.com/en/api/errors "Errors - Claude API - Anthropic") "Errors are always returned as JSON, with a top-level error object that always includes a type and message value." The shape is `{"type": "error", "error": {"type": "...", "message": "..."}}`. No `code` or `status` inside `error`.

**Gemini** — [*](https://blog.laozhang.ai/en/posts/gemini-api-error-troubleshooting "Gemini API Error Troubleshooting: Complete Guide to Fix 429, 400, 500 Errors (2026) | LaoZhang AI Blog") "Every Gemini API error returns a JSON body containing a status field (the gRPC status code like RESOURCE_EXHAUSTED) and a message field with human-readable details." Shape: `{"error": {"code": 400, "message": "...", "status": "INVALID_ARGUMENT", "details": [...]}}`.

So here's the key observation: **all three providers always return `{"error": <dict>}`** with `message` inside. That means the first branch (`isinstance(err, dict)`) handles all three. The other two branches in the code:

1. `elif err is not None` — for when `error` is a string, not a dict
2. `else` — for when there's no `error` key at all (e.g. `{"message": "...", "detail": "..."}`)

These are defensive fallbacks for edge cases like:
- Proxy/gateway errors that might return `{"error": "some string"}`
- OAuth-style errors with `{"error_description": "..."}`
- Plain text from a CDN/gateway (the `except Exception: raw = self.text` fallback)
- [*](https://github.com/googleapis/google-cloud-python/issues/14703 "Inconsistent Response Format for Gemini Model on Vertex AI (List vs. Dictionary) · Issue #14703 · googleapis/google-cloud-python") Vertex AI has been known to return lists instead of dicts in error responses, which would fall through to the text fallback

**Simplification potential**: The three identical extraction patterns (one per branch) are repetitive — they all try the same keys (`message`, `detail`, `type`, `status`, `code`). The question is whether you want to collapse the branches at the cost of losing the slightly different priority ordering per case. What do you think — worth exploring a refactor, or is the defensiveness justified?

<details class='token-usage-details'><summary>$0.4803</summary>

`total=190,624 | in=188,966 | out=1,658 | cached=83.0% | cache_new=31,869 | searches=3 | $0.4803`

</details>


All three providers use different SSE error event shapes:

- **OpenAI** ([docs](https://developers.openai.com/api/reference/resources/responses/streaming-events)): flat `{code, message}` — no nested `error` key, `code` is a string enum (e.g. `"rate_limit_exceeded"`)
- **Anthropic** ([docs](https://docs.anthropic.com/en/api/messages-streaming)): `{type: "error", error: {type, message}}` — no `code` or `status`
- **Gemini** ([docs](https://ai.google.dev/api/generate-content#method:-models.streamgeneratecontent)): `{error: {code, message, status, details}}` — `code` is numeric, `status` is the semantic gRPC code

The extraction logic:
1. `isinstance(event["error"], dict)` → use nested dict (Anthropic, Gemini)
2. `isinstance(event, dict)` but no nested `error` → use whole dict (OpenAI flat shape)
3. Not a dict at all → wrap as `{"message": stringified}` (defensive fallback)

**NB**: Unlike `error()`, this function does *not* promote Gemini's semantic `status` over numeric `code` — so Gemini returns `code=500` rather than `code="INTERNAL"`. This may be worth aligning.

In [ ]:
#| export
def _parse_sse_error_event(event):
    "Parse common SSE `error` event shapes to (message, error_type, code, raw)."
    raw = event
    e = event.get("error") if isinstance(event, dict) and isinstance(event.get("error"), dict) else (
        event if isinstance(event, dict) else {"message": _to_text(event)})
    msg = str(e.get("message") or e.get("detail") or _to_text(e))
    et = str(e.get("type") or e.get("status") or "")
    code = e.get("code")
    if (code is None or isinstance(code, int)) and isinstance(et, str) and et: code = et
    return AttrDict(message=msg, type=et, code=code, raw=raw)

In [ ]:
# OpenAI SSE — flat {code, message}, no nested "error" key
oai_sse = {"code": "rate_limit_exceeded", "message": "Rate limit exceeded"}

# Anthropic SSE — nested error dict with type+message
ant_sse = {"type": "error", "error": {"type": "overloaded_error", "message": "Overloaded"}}

# Gemini SSE — nested error dict with code(int)+message+status(string)
gem_sse = {"error": {"code": 500, "message": "An internal error has occurred", "status": "INTERNAL"}}

for name, evt in [("OpenAI", oai_sse), ("Anthropic", ant_sse), ("Gemini", gem_sse)]:
    error = _parse_sse_error_event(evt)
    print(f"{name:10s} | msg={error.message!r:45s} | et={error.type!r:25s} | code={error.code!r}")

OpenAI     | msg='Rate limit exceeded'                         | et=''                        | code='rate_limit_exceeded'
Anthropic  | msg='Overloaded'                                  | et='overloaded_error'        | code='overloaded_error'
Gemini     | msg='An internal error has occurred'              | et='INTERNAL'                | code=500


Do SSE events have a type like httpx.Response like we can patch?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

No — SSE events are just plain `dict`s after JSON parsing. There's no dedicated class like `httpx.Response` to patch onto. That's why `_parse_sse_error_event` is a standalone function rather than a `@patch` method.

If you wanted to make it patchable, you'd need to introduce a wrapper class for SSE events first — but that might be over-engineering it for what's essentially just a dict parser.

<details class='token-usage-details'><summary>$0.1729</summary>

`total=37,168 | in=36,770 | out=398 | cached=58.7% | cache_new=14,944 | $0.1729`

</details>


We do have an `SSE` class in `sse` dialog, maybe I'll patch it there later

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Makes sense — patching it onto the `SSE` class would keep it consistent with how `error()` works on `httpx.Response`. You'd also get the chance to add the Gemini `status`→`code` promotion that the note above mentioned is currently missing from the SSE version.

<details class='token-usage-details'><summary>$0.1461</summary>

`total=35,128 | in=35,006 | out=122 | cached=61.7% | cache_new=13,176 | $0.1461`

</details>


```py
@dataclass(frozen=True)
class SSEvent:
    "A parsed SSE event."
    event: Optional[str]
    data: str
    id: Optional[str] = None
    retry: Optional[int] = None
```

this is what it looks like

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice — since `SSEvent.data` is the raw string, you'd parse it with `json.loads(self.data)` first, then run the same extraction logic. And `frozen=True` means you'd use `@patch` rather than adding the method to the class directly.

One thing to note: `data` might not always be valid JSON (e.g. OpenAI sends `data: [DONE]` as a termination signal), so you'd want the same `try/except` guard as in `httpx.Response.error()`.

<details class='token-usage-details'><summary>$0.0343</summary>

`total=35,323 | in=35,196 | out=127 | cached=98.8% | cache_new=147 | $0.0343`

</details>


### `APIError`

Structured cross-provider API failure object.

**Why it exists**
- Callers need a single error shape across HTTP and stream failures from different providers.

**Design Notes**
- Includes normalized diagnostics (`status_code`, `code`, `error_type`, `request_id`, `retryable`) plus raw payload.
- Context fields (`provider`, `model`, `endpoint`) make failures actionable in multi-provider systems.

**Connections**
- Raised by transport/oapi/client layers and surfaced to high-level consumers.

In [ ]:
#| export
class APIError(FastLLMError):
    "Structured provider/API error with context."
    def __init__(self, message: str, *, provider: str = "", model: str = "", endpoint: str = "",
        status_code= None, error_type: str = "", code = None, request_id: str = "",
        retryable = None, raw = None):
        self.message = message or "API request failed"
        self.provider = provider or ""
        self.model = model or ""
        self.endpoint = endpoint or ""
        self.status_code = status_code
        self.error_type = error_type or ""
        self.code = code
        self.request_id = request_id or ""
        self.retryable = _retryable(status_code, self.error_type, self.code, self.message) if retryable is None else bool(retryable)
        self.raw = raw
        super().__init__(self.__str__())

    def with_context(self, *, provider: str = "", model: str = "", endpoint: str = "") -> "APIError":
        "Return a copy with missing context fields filled."
        return APIError(
            self.message,
            provider=self.provider or provider,
            model=self.model or model,
            endpoint=self.endpoint or endpoint,
            status_code=self.status_code,
            error_type=self.error_type,
            code=self.code,
            request_id=self.request_id,
            retryable=self.retryable,
            raw=self.raw,
        )

    def __str__(self):
        fields = ['message', 'provider', 'model', 'endpoint', 'status_code', 'error_type', 'code', 'request_id', 'retryable']
        parts = [f"{f}={getattr(self, f)!r}" for f in fields if getattr(self, f) not in (None, "", False)]
        return f"APIError({', '.join(parts)})"

    __repr__ = __str__

Can you walk me through this class and how it's been used in the fastllm repo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me start with the class structure, then we'll look at actual usage in the repo.

**`APIError` has three concerns:**

1. **Normalized diagnostics** — `__init__` captures a consistent set of fields regardless of provider:
   - `message`, `status_code`, `error_type`, `code` — from the parsed error
   - `request_id` — for provider support debugging
   - `retryable` — auto-classified via `_retryable()` if not explicitly passed
   - `raw` — the original payload for inspection

2. **Context enrichment** — `with_context()` returns a *copy* with `provider`/`model`/`endpoint` filled in. This is useful because the error might be created deep in transport code that doesn't know which provider it's talking to — the caller higher up can stamp that context on.

3. **Human-readable repr** — `__str__` builds a compact summary like: `anthropic /v1/messages model=claude-sonnet-4-20250514 status=401 type=authentication_error code=authentication_error: invalid x-api-key`

Let me look at how it's actually used in the repo:


<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;bash(&quot;rg \&#x27;APIError\&#x27; ~/aai-ws/fastllm_v2/…&#x27;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_01Csg9KtgFuK2cNpxBCaj4eG",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "bash(\"rg 'APIError' ~/aai-ws/fastllm_v2/fastllm_v2/ --no-heading -n\")"
    }
  },
  "result": "---------------------------------------------------------------------------\nOSError                                   Traceback (most recent call last)\nCell In[82], line 1\n----> 1 await maybe_await(pyrun(code='bash(\"rg \\'APIError\\' ~/aai-ws/fastllm_v2/fastllm_v2/ --no-heading -n\")'))\n\nFile ~/aai-ws/fastcore/fastcore/xtras.py:1033, in maybe_await(o)\n   1031 \"Await `o` if needed, and return it\"\n   1032 from inspect import isawaitable\n-> 1033 return await o if isawaitable(o) else o\n\nFile ~/aai-ws/safepyrun/safepyrun/core.py:317, in RunPython.__call__(self, code)\n    315 tb = e.__traceback__\n    316 while tb.tb_next: tb = tb.tb_next\n--> 317 raise e.with_traceback(tb) from None\n\nFile ~/aai-ws/safecmd/safecmd/core.py:32, in run(cmd, ignore_ex, split)\n     30 if res.stderr: out += res.stderr\n     31 if ignore_ex: return (res.returncode, out)\n---> 32 if res.returncode: raise IOError(out)\n     33 return out\n\nOSError: rg: /Users/keremturgutlu/aai-ws/fastllm_v2/fastllm_v2/: IO error for operation on /Users/keremturgutlu/aai-ws/fastllm_v2/fastllm_v2/: No such file or directory (os error 2)"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>bash(cmd=&#x27;ls ~/aai-ws/fastllm*/&#x27;)→&#x27;/Users/keremturgutlu/aai-ws/fastllm_old/…&#x27;</code></summary>

```json
{
  "id": "toolu_0151joHgSaTBRLmMxhZBAxYF",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "ls ~/aai-ws/fastllm*/"
    }
  },
  "result": "/Users/keremturgutlu/aai-ws/fastllm_old/:\n\u001b[1m\u001b[36mfastllm\u001b[m\u001b[m\n\u001b[1m\u001b[36mfastllm.egg-info\u001b[m\u001b[m\npyproject.toml\nREADME.md\n\u001b[1m\u001b[36mtests\u001b[m\u001b[m\n\n/Users/keremturgutlu/aai-ws/fastllm/:\ncachy.jsonl\n\u001b[1m\u001b[36mexamples\u001b[m\u001b[m\n\u001b[1m\u001b[36mfastllm\u001b[m\u001b[m\n\u001b[1m\u001b[36mfastllm_v2.egg-info\u001b[m\u001b[m\n\u001b[1m\u001b[36mfastllm.egg-info\u001b[m\u001b[m\n\u001b[1m\u001b[36mnbs\u001b[m\u001b[m\npyproject.toml\nREADME.md\n\u001b[1m\u001b[36mspecs\u001b[m\u001b[m\n\u001b[1m\u001b[36mtests\u001b[m\u001b[m\nTODO.md"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;bash(&quot;rg \&#x27;APIError\&#x27; ~/aai-ws/fastllm/fas…&#x27;)→&#x27;/Users/keremturgutlu/aai-ws/fastllm/fast…&#x27;</code></summary>

```json
{
  "id": "toolu_01NhsEd2FaYnrvde5TwPMQx5",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "bash(\"rg 'APIError' ~/aai-ws/fastllm/fastllm/ --no-heading -n\")"
    }
  },
  "result": "/Users/keremturgutlu/aai-ws/fastllm/fastllm/__init__.py:7:from .errors import APIError, FastLLMError, ProtocolError, SpecError, UnsupportedCapabilityError\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/__init__.py:16:__all__ = \"__version__ FastLLMError APIError UnsupportedCapabilityError ProtocolError SpecError OpSpec OpenAPIClient spec_to_ops load_spec_json load_spec_yaml load_spec_file load_spec_url merge_ops openai_ops anthropic_ops gemini_ops client_from_spec Part Msg ToolCall Usage Caps RequestOptions Completion Delta OpenAIClient AnthropicClient GeminiClient ModelPrice CostBreakdown estimate_cost infer_provider mk_auto_client acompletion StreamSummary acollect_stream FileRef to_input_file_part afile_create afile_get afile_list afile_delete afile_content\".split()\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/clients.py:15:from .errors import APIError, UnsupportedCapabilityError, api_error_from_http\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/clients.py:549:    \"Raise structured APIError for HTTP/provider failures with context.\"\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/clients.py:550:    if isinstance(exc, APIError):\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:106:class APIError(FastLLMError):\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:123:    def with_context(self, *, provider: str = \"\", model: str = \"\", endpoint: str = \"\") -> \"APIError\":\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:125:        return APIError(\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:152:def api_error_from_http(exc: httpx.HTTPStatusError, *, provider: str = \"\", model: str = \"\", endpoint: str = \"\") -> APIError:\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:153:    \"Build APIError from httpx HTTPStatusError.\"\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:159:    return APIError(\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:172:def api_error_from_event(event: Any, *, provider: str = \"\", model: str = \"\", endpoint: str = \"\") -> APIError:\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:173:    \"Build APIError from provider SSE/event-level error payload.\"\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/errors.py:175:    return APIError(msg, provider=provider, model=model, endpoint=endpoint, error_type=et, code=code, raw=raw)\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/highlevel.py:15:from .errors import APIError\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/highlevel.py:326:    if isinstance(e, APIError):\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/oapi.py:13:from .errors import APIError, UnsupportedCapabilityError, api_error_from_http\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/oapi.py:204:        \"Raise APIError with operation context for dynamic op calls.\"\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/oapi.py:208:        if isinstance(exc, APIError):\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/files.py:13:from .errors import APIError, UnsupportedCapabilityError, api_error_from_http\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/files.py:247:    except APIError:\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/files.py:282:    except APIError:\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/files.py:316:    except APIError:\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/files.py:351:    except APIError:\n/Users/keremturgutlu/aai-ws/fastllm/fastllm/files.py:392:    except APIError:"
}
```

</details>

Here's how `APIError` flows through the codebase — there are **three creation points** and **two consumption patterns**:

**Creation points:**

1. **`api_error_from_http`** (errors.py:152) — wraps `httpx.HTTPStatusError` → calls `resp.error()` to parse, then builds `APIError`
2. **`api_error_from_event`** (errors.py:172) — wraps SSE error events → calls `_parse_sse_error_event`, then builds `APIError`
3. **`_raise_for_error`** in clients.py:549 — catches raw exceptions, delegates to the above two, and stamps `provider`/`model`/`endpoint` context

**Consumption patterns:**

1. **Retry logic** — `highlevel.py` checks `isinstance(e, APIError)` and looks at `e.retryable` to decide whether to retry
2. **Graceful fallback** — `files.py` catches `APIError` in multiple places (file create/get/list/delete/content) for handling file operations that may fail

The key architectural insight: errors are created low (transport) and enriched as they bubble up via `with_context()`. So transport code doesn't need to know which provider it's talking to — the client layer stamps that on.

Want to dig into any of these specific usage sites?

<details class='token-usage-details'><summary>$0.2883</summary>

`total=143,371 | in=142,466 | out=905 | cached=87.3% | cache_new=13,801 | $0.2883`

</details>


What about using `__repr__` instead of `__str__` ? Also maybe we can create a tabular version using `__repr_markdown__` for solveit :)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good ideas! Here's the tradeoff:

- **`__str__`** is what Python uses in tracebacks and `print()` — so the current compact one-liner format is actually ideal there. You don't want a table in a traceback.
- **`__repr__`** is for the REPL/debugger — currently it falls back to `__str__`. You could make it more explicit, e.g. `APIError(provider='anthropic', status_code=401, ...)`.
- **`__repr_markdown__`** would give you a nice rich display when inspecting an error in solveit — a table would be great for that.

So the best approach is probably to keep `__str__` as-is, add a proper `__repr__`, and add `__repr_markdown__` for the rich table. Want to sketch these out?

<details class='token-usage-details'><summary>$0.0777</summary>

`total=37,801 | in=37,364 | out=437 | cached=89.5% | cache_new=3,659 | $0.0777`

</details>


Sounds good

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

For `__repr__`, you'd want to show the constructor call with the key fields — something like `APIError(message=..., provider=..., status_code=...)`. You can filter to only non-empty/non-None fields to keep it clean.

Want to take a crack at writing `__repr__` first, and then we'll do the markdown table one?

<details class='token-usage-details'><summary>$0.0381</summary>

`total=37,780 | in=37,623 | out=157 | cached=98.6% | cache_new=294 | $0.0381`

</details>


Do them both

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here you go:

```python
def __repr__(self):
    fields = ['message', 'provider', 'model', 'endpoint', 'status_code', 'error_type', 'code', 'request_id', 'retryable']
    parts = [f"{f}={getattr(self, f)!r}" for f in fields if getattr(self, f) not in (None, "", False)]
    return f"APIError({', '.join(parts)})"

def __repr_markdown__(self):
    rows = [
        ("message", self.message), ("provider", self.provider), ("model", self.model),
        ("endpoint", self.endpoint), ("status_code", self.status_code), ("error_type", self.error_type),
        ("code", self.code), ("request_id", self.request_id), ("retryable", self.retryable),
    ]
    rows = [(k, v) for k, v in rows if v not in (None, "", False)]
    header = "| Field | Value |\n|---|---|"
    body = "\n".join(f"| **{k}** | `{v}` |" for k, v in rows)
    return f"{header}\n{body}"
```

A couple of notes:
- Both filter out empty/None/False fields to keep output clean
- `retryable=False` is filtered since that's the "normal" case — only `retryable=True` is noteworthy
- `raw` is excluded from both since it can be very large — you can always inspect it directly

<details class='token-usage-details'><summary>$0.0470</summary>

`total=38,192 | in=37,766 | out=426 | cached=99.1% | cache_new=144 | $0.0470`

</details>


Ok let's test it now

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;err = APIError(&quot;invalid x-api-key&quot;, prov…&#x27;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_0178P4ErQqC9cyPx1y6nBR3Y",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "err = APIError(\"invalid x-api-key\", provider=\"anthropic\", model=\"claude-sonnet-4-20250514\",\n               endpoint=\"/v1/messages\", status_code=401, error_type=\"authentication_error\",\n               code=\"authentication_error\", request_id=\"req_011CZarRA1XwDwqX5DDehf2N\")\nprint(repr(err))\nprint('---')\nprint(err.__repr_markdown__())"
    }
  },
  "result": "---------------------------------------------------------------------------\nPermissionError                           Traceback (most recent call last)\nCell In[90], line 1\n----> 1 await maybe_await(pyrun(code='err = APIError(\"invalid x-api-key\", provider=\"anthropic\", model=\"claude-sonnet-4-20250514\",\\n               endpoint=\"/v1/messages\", status_code=401, error_type=\"authentication_error\",\\n               code=\"authentication_error\", request_id=\"req_011CZarRA1XwDwqX5DDehf2N\")\\nprint(repr(err))\\nprint(\\'---\\')\\nprint(err.__repr_markdown__())'))\n\nFile ~/aai-ws/fastcore/fastcore/xtras.py:1033, in maybe_await(o)\n   1031 \"Await `o` if needed, and return it\"\n   1032 from inspect import isawaitable\n-> 1033 return await o if isawaitable(o) else o\n\nFile ~/aai-ws/safepyrun/safepyrun/core.py:317, in RunPython.__call__(self, code)\n    315 tb = e.__traceback__\n    316 while tb.tb_next: tb = tb.tb_next\n--> 317 raise e.with_traceback(tb) from None\n\nFile ~/aai-ws/safepyrun/safepyrun/core.py:157, in _ReadOnlyCallable.__call__(self, *args, **kwargs)\n    155 def __call__(self, *args, **kwargs):\n    156     n = object.__getattribute__(self, '_name')\n--> 157     if n: raise PermissionError(f\"Calling `{n}` is not permitted; use `allow()` to add it\") from None\n    158     raise PermissionError(f\"Calling {type(object.__getattribute__(self, '_obj')).__name__} not allowed in sandbox\")\n\nPermissionError: Calling `APIError` is not permitted; use `allow()` to add it"
}
```

</details>

`APIError` isn't whitelisted for `pyrun`. You'll need to run this directly in a code cell instead:

```python
err = APIError("invalid x-api-key", provider="anthropic", model="claude-sonnet-4-20250514",
               endpoint="/v1/messages", status_code=401, error_type="authentication_error",
               code="authentication_error", request_id="req_011CZarRA1XwDwqX5DDehf2N")
repr(err)
```

And then in a second cell:

```python
err
```

That second one should trigger the markdown table rendering in solveit.

<details class='token-usage-details'><summary>$0.1227</summary>

`total=78,118 | in=77,801 | out=317 | cached=91.8% | cache_new=5,109 | $0.1227`

</details>


No I meant with actual api calls like we've been doing so far

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right! You can reuse one of those fake-key calls. Try this in a code cell:

```python
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hi"}]}
)
e = resp.error()
err = APIError(e.message, provider="anthropic", model="claude-sonnet-4-20250514",
               endpoint="/v1/messages", status_code=resp.status_code,
               error_type=e.type, code=e.code, request_id=_req_id(resp.headers), raw=e.raw)
err
```

That should trigger `__repr_markdown__` and show the table.

<details class='token-usage-details'><summary>$0.0615</summary>

`total=39,986 | in=39,459 | out=527 | cached=96.3% | cache_new=1,236 | $0.0615`

</details>


In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hi"}]}
)
e = resp.error()
err = APIError(e.message, provider="anthropic", model="claude-sonnet-4-20250514",
               endpoint="/v1/messages", status_code=resp.status_code,
               error_type=e.type, code=e.code, request_id=_req_id(resp.headers), raw=e.raw)
err

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='/v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')

In [ ]:
print(err)

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='/v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')


I've simplified it by making __repl__ = __str__ 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looks clean! Two things I notice from the output:

1. **`status_code=200`** — that seems wrong for an auth error. Anthropic should return 401. Are you sure `resp.status_code` is giving the right value? (Maybe check `resp.status_code` directly.)

2. **`__repr_markdown__`** — did you add that too? The output is showing the `__repr__` format, not the table.

<details class='token-usage-details'><summary>$0.2028</summary>

`total=40,386 | in=39,971 | out=415 | cached=54.0% | cache_new=17,736 | $0.2028`

</details>


In [ ]:
resp.text

'{"type":"error","error":{"type":"authentication_error","message":"invalid x-api-key"},"request_id":"req_011CZawgT2EDaKt1J84fwjww"}'

In [ ]:
resp.status_code

401

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's odd — a 200 status code with an error body. This might be a caching artifact from `enable_cachy()` — the original response was likely 401, but the cache may not be preserving the status code correctly. You could try disabling cachy temporarily and re-running the request to confirm.

<details class='token-usage-details'><summary>$0.0473</summary>

`total=40,529 | in=40,273 | out=256 | cached=97.7% | cache_new=591 | $0.0473`

</details>


Good thinking, I'll change the message slightly to invalidate cache

In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "hello!"}]}
)
e = resp.error()
err = APIError(e.message, provider="anthropic", model="claude-sonnet-4-20250514",
               endpoint="/v1/messages", status_code=resp.status_code,
               error_type=e.type, code=e.code, request_id=_req_id(resp.headers), raw=e.raw)
err

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='/v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`status_code=401` now — confirmed it was a cachy artifact. The `__repr__` looks good and compact.

Did you still want to add `__repr_markdown__` for the table view in solveit?

<details class='token-usage-details'><summary>$0.0386</summary>

`total=40,795 | in=40,740 | out=55 | cached=98.0% | cache_new=267 | $0.0386`

</details>


I am seeing the `__repr__` version but was expecting `__repr_markdown__`, you can check solveit source code to figure this out

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check how solveit handles output rendering:


<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;rg &#x27;__repr_markdown__&#x27; ~/aai-ws/solveit/…&quot;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_01Y7zpPCMm76Krnh4ZW9TbBJ",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "rg '__repr_markdown__' ~/aai-ws/solveit/ --no-heading -n -l"
    }
  },
  "result": "---------------------------------------------------------------------------\nOSError                                   Traceback (most recent call last)\nCell In[93], line 1\n----> 1 await maybe_await(bash(cmd=\"rg '__repr_markdown__' ~/aai-ws/solveit/ --no-heading -n -l\"))\n\nFile ~/aai-ws/safecmd/safecmd/core.py:326, in bash(cmd, as_dict, rm_cmds, rm_dests)\n    312 def bash(\n    313     cmd:str,  # Bash command string to execute - all shell features like pipes and subcommands are supported\n    314     as_dict:bool=False, # Return a dict response with 'success' or 'error' key\n    315     rm_cmds:str=None,  # Temp remove these commands from allow list\n    316     rm_dests:str=None  # Temp remove these destinations from allow list\n    317 ): # dict with 'success' or 'error' key; value is stdout+stderr for success, or error message otherwise\n    318     \"\"\"Run a bash shell command line safely and return the concatencated stdout and stderr.\n    319     Since it is run with `bash`, special chars like `$` and `*` are handled by the shell, so must be quoted if literal.\n    320     `cmd` is parsed and all calls are checked against an allow-list.\n   (...)    324     All operators are supported. Output redirects are validated against allowed destinations (default: ./ and /tmp).\n    325     rm_ params are comma-separated strs.\"\"\"\n--> 326     try: res = safe_run(cmd, rm_cmds=rm_cmds, rm_dests=rm_dests)\n    327     except PermissionError as e:\n    328         eff_cmds, eff_dests = _eff_sets(rm_cmds=rm_cmds, rm_dests=rm_dests)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:309, in safe_run(cmd, cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests, ignore_ex, split)\n    307 eff_cmds, eff_dests = _eff_sets(cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests)\n    308 validate(cmd, eff_cmds, eff_dests)\n--> 309 return run(cmd, ignore_ex=ignore_ex, split=split)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:32, in run(cmd, ignore_ex, split)\n     30 if res.stderr: out += res.stderr\n     31 if ignore_ex: return (res.returncode, out)\n---> 32 if res.returncode: raise IOError(out)\n     33 return out\n\nOSError:"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;rg &#x27;__repr_markdown__&#x27; ~/aai-ws/solveit/…&quot;)→&#x27;No output&#x27;</code></summary>

```json
{
  "id": "toolu_01LNRbc718SJ85FtcXXVKGLX",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "rg '__repr_markdown__' ~/aai-ws/solveit/ --no-heading -n -l 2>&1 || true"
    }
  },
  "result": "No output"
}
```

</details>

No results in solveit itself. Let me check the ipykernel/IPython side — that's where display formatting happens:


<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;rg &#x27;_repr_markdown_&#x27; ~/aai-ws/ipykernel-…&quot;)→&#x27;/Users/keremturgutlu/aai-ws/ipykernel-he…&#x27;</code></summary>

```json
{
  "id": "toolu_011zFFcPGrQ76igF8guv6Ha9",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "rg '_repr_markdown_' ~/aai-ws/ipykernel-helper/ --no-heading -n 2>&1 || true"
    }
  },
  "result": "/Users/keremturgutlu/aai-ws/ipykernel-helper/ipykernel_helper/core.py:334:def _repr_markdown_(self:Markdown):\n/Users/keremturgutlu/aai-ws/ipykernel-helper/ipykernel_helper/_modidx.py:34:                                       'ipykernel_helper.core.Markdown._repr_markdown_': ( 'core.html#markdown._repr_markdown_',\n/Users/keremturgutlu/aai-ws/ipykernel-helper/nbs/00_core.ipynb:1793:    \"def _repr_markdown_(self:Markdown):\\n\","
}
```

</details>

Interesting — IPython uses `_repr_markdown_` (with leading underscore and trailing underscore), not `__repr_markdown__` (double underscores). The method should be:

```python
def _repr_markdown_(self):
```

not:

```python
def __repr_markdown__(self):
```

Update the `@patch` to use `_repr_markdown_` and it should render as a markdown table in solveit.

<details class='token-usage-details'><summary>$0.1765</summary>

`total=163,855 | in=163,410 | out=445 | cached=97.0% | cache_new=2,334 | $0.1765`

</details>


### `api_error_from_http`

Converts `httpx.HTTPStatusError` into `APIError`.

**Why it exists**
- HTTP error payloads differ by provider and endpoint.

**Design Notes**
- Parses best-effort structured fields and computes retryability hints.

**Connections**
- Core adapter used by transport/oapi/client error paths.

In [ ]:
#| export
@patch
def api_error(self:httpx.HTTPStatusError, *, provider: str = "", model: str = "", endpoint: str = ""):
    "Build APIError from httpx HTTPStatusError."
    resp = self.response
    err = resp.error()
    if not endpoint and resp.request is not None:
        endpoint = f"{resp.request.method.upper()} {resp.request.url.path}"
    return APIError(
        err.message,
        provider=provider,
        model=model,
        endpoint=endpoint,
        status_code=resp.status_code,
        error_type=err.type,
        code=err.code,
        request_id=_req_id(resp.headers),
        raw=err.raw,
    )

Let's test this function now

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;print(doc(api_error_from_http))&#x27;)→&#x27;&lt;function api_error_from_http at 0x30144…&#x27;</code></summary>

```json
{
  "id": "toolu_01QURQk6ezksEJdv6t1xGK7b",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "print(doc(api_error_from_http))"
    }
  },
  "result": "<function api_error_from_http at 0x30144a660>"
}
```

</details>

Hmm, `doc()` didn't show the signature — but we can see it above. The function takes an `httpx.HTTPStatusError`, so you need to trigger one with `raise_for_status()`. Try this:

```python
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "test api_error_from_http"}]}
)
try: resp.raise_for_status()
except httpx.HTTPStatusError as exc:
    err = api_error_from_http(exc, provider="anthropic", model="claude-sonnet-4-20250514")
    print(err)
```

<details class='token-usage-details'><summary>$0.4445</summary>

`total=85,126 | in=84,834 | out=292 | cached=49.3% | cache_new=42,367 | $0.4445`

</details>


In [ ]:
resp = httpx.post(
    "https://api.anthropic.com/v1/messages",
    headers={"x-api-key": "sk-fake123", "anthropic-version": "2023-06-01"},
    json={"model": "claude-sonnet-4-20250514", "max_tokens": 10, "messages": [{"role": "user", "content": "test api_error_from_http"}]}
)
try: resp.raise_for_status()
except httpx.HTTPStatusError as exc: print(exc.api_error(provider="anthropic", model="claude-sonnet-4-20250514"))

APIError(message='invalid x-api-key', provider='anthropic', model='claude-sonnet-4-20250514', endpoint='POST /v1/messages', status_code=401, error_type='authentication_error', code='authentication_error')


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `@patch def error(self: httpx.Response)` from earlier in the notebook hasn't been run in the current kernel — so `resp.error()` isn't available. You'll need to re-run that cell first (the one with `@patch def error(self: httpx.Response)`), then retry.

<details class='token-usage-details'><summary>$0.2196</summary>

`total=42,838 | in=42,623 | out=215 | cached=50.7% | cache_new=20,052 | $0.2196`

</details>


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Cachy strikes again — status 200 with an error body means `raise_for_status()` doesn't fire, so `api_error_from_http` never gets called.

You'll need to either disable cachy for this test, or hit a provider that cachy hasn't cached yet. Alternatively, you could manually construct the `HTTPStatusError` to test the function directly:

```python
exc = httpx.HTTPStatusError("auth error", request=resp.request, response=resp)
resp._status_code = 401  # override cached status
api_error_from_http(exc, provider="anthropic", model="claude-sonnet-4-20250514")
```

<details class='token-usage-details'><summary>$0.2187</summary>

`total=42,824 | in=42,665 | out=159 | cached=50.6% | cache_new=20,551 | $0.2187`

</details>


Perhaps cachy doesn't preserve the status codes? Can you check the source code

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;rg &#x27;status_code\|status&#x27; ~/aai-ws/cachy/…&quot;)→&#x27;----------------------------------------…&#x27;</code></summary>

```json
{
  "id": "toolu_01PFuPgJSAUSxKTTbrZWoFfp",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "rg 'status_code\\|status' ~/aai-ws/cachy/cachy/ --no-heading -n"
    }
  },
  "result": "---------------------------------------------------------------------------\nOSError                                   Traceback (most recent call last)\nCell In[53], line 1\n----> 1 await maybe_await(bash(cmd=\"rg 'status_code\\\\|status' ~/aai-ws/cachy/cachy/ --no-heading -n\"))\n\nFile ~/aai-ws/safecmd/safecmd/core.py:326, in bash(cmd, as_dict, rm_cmds, rm_dests)\n    312 def bash(\n    313     cmd:str,  # Bash command string to execute - all shell features like pipes and subcommands are supported\n    314     as_dict:bool=False, # Return a dict response with 'success' or 'error' key\n    315     rm_cmds:str=None,  # Temp remove these commands from allow list\n    316     rm_dests:str=None  # Temp remove these destinations from allow list\n    317 ): # dict with 'success' or 'error' key; value is stdout+stderr for success, or error message otherwise\n    318     \"\"\"Run a bash shell command line safely and return the concatencated stdout and stderr.\n    319     Since it is run with `bash`, special chars like `$` and `*` are handled by the shell, so must be quoted if literal.\n    320     `cmd` is parsed and all calls are checked against an allow-list.\n   (...)    324     All operators are supported. Output redirects are validated against allowed destinations (default: ./ and /tmp).\n    325     rm_ params are comma-separated strs.\"\"\"\n--> 326     try: res = safe_run(cmd, rm_cmds=rm_cmds, rm_dests=rm_dests)\n    327     except PermissionError as e:\n    328         eff_cmds, eff_dests = _eff_sets(rm_cmds=rm_cmds, rm_dests=rm_dests)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:309, in safe_run(cmd, cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests, ignore_ex, split)\n    307 eff_cmds, eff_dests = _eff_sets(cmds, dests, add_cmds, add_dests, rm_cmds, rm_dests)\n    308 validate(cmd, eff_cmds, eff_dests)\n--> 309 return run(cmd, ignore_ex=ignore_ex, split=split)\n\nFile ~/aai-ws/safecmd/safecmd/core.py:32, in run(cmd, ignore_ex, split)\n     30 if res.stderr: out += res.stderr\n     31 if ignore_ex: return (res.returncode, out)\n---> 32 if res.returncode: raise IOError(out)\n     33 return out\n\nOSError:"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>bash(cmd=&quot;rg &#x27;status&#x27; ~/aai-ws/cachy/cachy/core.py…&quot;)→&#x27;52:            return httpx.Response(sta…&#x27;</code></summary>

```json
{
  "id": "toolu_016EdHsVMXz3PBVemrofYrUv",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "rg 'status' ~/aai-ws/cachy/cachy/core.py --no-heading -n"
    }
  },
  "result": "52:            return httpx.Response(status_code=200, content=res['response'], headers=res.get('headers'), request=r)\n58:        return httpx.Response(status_code=res.status_code, content=content, headers=headers, request=r)\n69:            return httpx.Response(status_code=200, content=res['response'], headers=res.get('headers'), request=r)\n75:        return httpx.Response(status_code=res.status_code, content=content, headers=headers, request=r)"
}
```

</details>

There it is — line 52 and 69. When cachy has a cache hit, it constructs `httpx.Response(status_code=200, ...)` — hardcoded 200 regardless of what the original status code was. The non-cached path (lines 58, 75) correctly preserves `res.status_code`.

So cachy needs to store and restore the original status code in the cached entry. Looks like the cache dict has `response` and `headers` but no `status_code` field.

<details class='token-usage-details'><summary>$0.1288</summary>

`total=130,787 | in=130,509 | out=278 | cached=97.9% | cache_new=829 | $0.1288`

</details>


Let's fix it then can you share the link to the dialog

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details'>
<summary><code>pyrun(code=&quot;dialog_link(&#x27;/aai-ws/cachy/nbs/00_core&#x27;)&quot;)→&#x27;&lt;a href=&quot;/dialog_?name=aai-ws%2Fcachy%2F…&#x27;</code></summary>

```json
{
  "id": "toolu_01G9Swf4wT3g8UEpsr85aeRT",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "dialog_link('/aai-ws/cachy/nbs/00_core')"
    }
  },
  "result": "<a href=\"/dialog_?name=aai-ws%2Fcachy%2Fnbs%2F00_core\" target=\"_blank\">aai-ws/cachy/nbs/00_core</a>"
}
```

</details>

Here's the dialog: <a href="/dialog_?name=aai-ws%2Fcachy%2Fnbs%2F00_core" target="_blank">aai-ws/cachy/nbs/00_core</a>

The fix is on lines 52 and 69 — instead of hardcoding `status_code=200`, cachy needs to:

1. **Store** the original `status_code` when caching the response
2. **Restore** it when constructing the cached `httpx.Response`

Want me to find the exact cells that need changing?

<details class='token-usage-details'><summary>$0.0889</summary>

`total=88,550 | in=88,339 | out=211 | cached=98.0% | cache_new=1,431 | $0.0889`

</details>


Can you tell me what changes to make with code snippets, I'll work directly in the dialog

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me first check how the cache stores responses so I can give you the right fix:


<details class='tool-usage-details'>
<summary><code>bash(cmd=&#x27;cat -n ~/aai-ws/cachy/cachy/core.py&#x27;)→&#x27;1\t&quot;&quot;&quot;Cache your API calls with a single …&#x27;</code></summary>

```json
{
  "id": "toolu_0157dWgXxw2dazT1s2ijkBZS",
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "cat -n ~/aai-ws/cachy/cachy/core.py"
    }
  },
  "result": "1\t\"\"\"Cache your API calls with a single line of code. No mocks, no fixtures. Just faster, cleaner code.\"\"\"\n     2\t\n     3\t# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/00_core.ipynb.\n     4\t\n     5\t# %% auto #0\n     6\t__all__ = ['doms', 'disable_cachy', 'enable_cachy']\n     7\t\n     8\t# %% ../nbs/00_core.ipynb #fa68c98b\n     9\timport hashlib,httpx,json\n    10\tfrom fastcore.utils import *\n    11\t\n    12\t# %% ../nbs/00_core.ipynb #527e76df\n    13\tdoms = (\"api.openai.com\", \"api.anthropic.com\", \"generativelanguage.googleapis.com\", \"api.deepseek.com\")\n    14\t\n    15\t# %% ../nbs/00_core.ipynb #09571365\n    16\tdef _should_cache(url, doms): return any(dom in str(url) for dom in doms)\n    17\t\n    18\t# %% ../nbs/00_core.ipynb #6924d508\n    19\tdef _cache(key, cfp):\n    20\t    with open(cfp, \"r\") as f:\n    21\t        line = first(f, lambda l: json.loads(l)[\"key\"] == key)\n    22\t        return json.loads(line) if line else None\n    23\t\n    24\t# %% ../nbs/00_core.ipynb #72a0213b\n    25\tdef _write_cache(key, content, cfp, hdrs):\n    26\t    with open(cfp, \"a\") as f: f.write(json.dumps({\"key\":key, \"response\": content, \"headers\":hdrs})+\"\\n\")\n    27\t\n    28\t# %% ../nbs/00_core.ipynb #3c7fca19\n    29\tdef _content(r):\n    30\t    \"Extract content from request.\"\n    31\t    if not hasattr(r, '_content'): r.read()\n    32\t    boundary = httpx._multipart.get_multipart_boundary_from_content_type(r.headers.get(\"Content-Type\", \"\").encode())\n    33\t    return r.content.replace(boundary, b\"cachy-boundary\") if boundary else r.content\n    34\t\n    35\t# %% ../nbs/00_core.ipynb #a69ba22c\n    36\tdef _key(r, is_stream=False):\n    37\t    \"Create a unique, deterministic id from the request `r`.\"\n    38\t    return hashlib.sha256(f\"{r.url.copy_remove_param('key')}{is_stream}\".encode() + _content(r)).hexdigest()[:8]\n    39\t\n    40\t# %% ../nbs/00_core.ipynb #25ced1ba\n    41\tdef _res_hdrs(res, hdrs=None): return {k: v for k, v in res.headers.items() if k.lower() in hdrs} if hdrs else None\n    42\t\n    43\t# %% ../nbs/00_core.ipynb #37ad35fa\n    44\tdef _apply_async_patch(cfp, doms, hdrs, debug=False):    \n    45\t    @patch\n    46\t    async def send(self:httpx._client.AsyncClient, r, **kwargs):\n    47\t        is_stream = kwargs.get(\"stream\")\n    48\t        if not _should_cache(r.url, doms): return await self._orig_send(r, **kwargs)\n    49\t        key = _key(r, is_stream=False)\n    50\t        if res := _cache(key,cfp):\n    51\t            if debug: print(f\"\ud83d\udfe2 HIT {key}\\n{r.content}\")\n    52\t            return httpx.Response(status_code=200, content=res['response'], headers=res.get('headers'), request=r)\n    53\t        res = await self._orig_send(r, **kwargs)\n    54\t        content = res.read().decode() if not is_stream else b''.join([c async for c in res.aiter_bytes()]).decode()\n    55\t        if debug: print(f\"\ud83d\udd34 MISS {key}\\n{r.content}\")\n    56\t        headers = _res_hdrs(res, hdrs)\n    57\t        _write_cache(key,content,cfp,headers)\n    58\t        return httpx.Response(status_code=res.status_code, content=content, headers=headers, request=r)\n    59\t\n    60\t# %% ../nbs/00_core.ipynb #950030b0\n    61\tdef _apply_sync_patch(cfp, doms, hdrs, debug=False):    \n    62\t    @patch\n    63\t    def send(self:httpx._client.Client, r, **kwargs):\n    64\t        is_stream = kwargs.get(\"stream\")\n    65\t        if not _should_cache(r.url, doms): return self._orig_send(r, **kwargs)\n    66\t        key = _key(r, is_stream=False)\n    67\t        if res := _cache(key,cfp):\n    68\t            if debug: print(f\"\ud83d\udfe2 HIT {key}\\n{r.content}\")\n    69\t            return httpx.Response(status_code=200, content=res['response'], headers=res.get('headers'), request=r)\n    70\t        res = self._orig_send(r, **kwargs)\n    71\t        content = res.read().decode() if not is_stream else b''.join(list(res.iter_bytes())).decode()\n    72\t        if debug: print(f\"\ud83d\udd34 MISS {key}\\n{r.content}\")\n    73\t        headers = _res_hdrs(res, hdrs)\n    74\t        _write_cache(key,content,cfp,headers)\n    75\t        return httpx.Response(status_code=res.status_code, content=content, headers=headers, request=r)\n    76\t\n    77\t# %% ../nbs/00_core.ipynb #3be9fe57\n    78\tdef disable_cachy():\n    79\t    if not hasattr(httpx._client.AsyncClient, '_orig_send'): return\n    80\t    httpx._client.AsyncClient.send = httpx._client.AsyncClient._orig_send\n    81\t    httpx._client.Client.send      = httpx._client.Client._orig_send\n    82\t\n    83\t# %% ../nbs/00_core.ipynb #a8d53f6a\n    84\tdef enable_cachy(cache_dir=None, doms=doms, hdrs=None, debug=False):\n    85\t    disable_cachy()\n    86\t    if hdrs is None: hdrs=[]\n    87\t    cfp = Path(cache_dir or find_file_parents(\"pyproject.toml\") or \".\") / \"cachy.jsonl\"\n    88\t    cfp.touch(exist_ok=True)   \n    89\t    _apply_sync_patch(cfp, doms, hdrs, debug)\n    90\t    _apply_async_patch(cfp, doms, hdrs, debug)"
}
```

</details>

Two places to change:

**1. `_write_cache` (line 26)** — store `status_code`:
```python
def _write_cache(key, content, cfp, hdrs, status_code=200):
    with open(cfp, "a") as f: f.write(json.dumps({"key":key, "response": content, "headers":hdrs, "status_code":status_code})+"\n")
```

**2. Cache hit lines (52 and 69)** — read it back instead of hardcoding 200:
```python
return httpx.Response(status_code=res.get('status_code', 200), content=res['response'], headers=res.get('headers'), request=r)
```

**3. Cache miss lines (57–58 and 73–75)** — pass `status_code` to `_write_cache`:
```python
_write_cache(key, content, cfp, headers, status_code=res.status_code)
```

The `get('status_code', 200)` default ensures backward compatibility with existing cache entries that don't have the field.

<details class='token-usage-details'><summary>$0.1120</summary>

`total=91,683 | in=91,083 | out=600 | cached=96.7% | cache_new=571 | $0.1120`

</details>


Ok done, have a look at the results now - do you still see any 200s?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details'>
<summary><code>pyrun(code=&#x27;find_msgs(dname=&quot;/aai-ws/cachy/nbs/00_co…&#x27;)→&quot;[{&#x27;id&#x27;: &#x27;_72a0213b&#x27;, &#x27;time_run&#x27;: &#x27;2026-0…&quot;</code></summary>

```json
{
  "id": "toolu_01RWgFcsNuEKhkpbWYM5QBwy",
  "call": {
    "function": "pyrun",
    "arguments": {
      "code": "find_msgs(dname=\"/aai-ws/cachy/nbs/00_core\", re_pattern=\"_write_cache|status_code=200|status_code=res\")"
    }
  },
  "result": "<TRUNCATED>\u202672a0213b', 'time_run': '2026-03-31T09:45:04.998768+00:00', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': '#| exports\\ndef _write_cache(key, content, cfp, hdrs, status_code=200):\\n    with open(cfp, \"a\") as f: f.write(json.dumps({\"key\":key, \"response\": content, \"headers\":hdrs, \"status_code\":status_code})+\"\\\\n\")', 'output': '', 'msg_type': 'code', 'input_tokens': 124, 'output_tokens': {}}, {'id': '_d802ec92', 'time_run': '2026-03-31T09:45:05.116216+00:00', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': '@patch\\ndef send(self:httpx._client.Client, r, **kwargs):\\n    if not _should_cache(r.url, doms): return self._orig_send(r, **kwargs)\\n    key = key(r)\\n    if res := _cache(key,\"cachy.jsonl\"): return httpx.Response(status_code=200, content=res, request=r)\\n    res = self._orig_send(r, **kwargs)\\n    content = res.read().decode()\\n    _write_cache(key, content, \"cachy.jsonl\")\\n    return httpx.Response(status_code=res.status_code, content=content, request=r)', 'output': '', 'msg_type': 'code', 'input_tokens': 208, 'output_tokens': {}}, {'id': '_30aa118e', 'time_run': '2026-03-31T09:45:05.695837+00:00', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': '@patch\\ndef send(self:httpx._client.Client, r, **kwargs):\\n    is_stream = kwargs.get(\"stream\")\\n    if not _should_cache(r.url, doms): return self._orig_send(r, **kwargs)\\n    key = _key(r, is_stream=False)\\n    if res := _cache(key,\"cachy.jsonl\"): return httpx.Response(status_code=200, content=res, request=r)\\n    res = self._orig_send(r, **kwargs)\\n    content = res.read().decode() if not is_stream else b\\'\\'.join(list(res.iter_bytes())).decode()\\n    _write_cache(key, content, \"cachy.jsonl\")\\n    return httpx.Response(status_code=res.status_code, content=content, request=r)', 'output': '', 'msg_type': 'code', 'input_tokens': 249, 'output_tokens': {}}, {'id': '_056e9c1e', 'time_run': '2026-03-31T09:45:05.749994+00:00', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': 'def _apply_patch():    \\n    @patch    \\n    def send(self:httpx._client.Client, r, **kwargs):\\n        is_stream = kwargs.get(\"stream\")\\n        if not _should_cache(r.url, doms): return self._orig_send(r, **kwargs)\\n        key = _key(r, is_stream=False)\\n        if res := _cache(key,\"cachy.jsonl\"): return httpx.Response(status_code=200, content=res, request=r)\\n        res = self._orig_send(r, **kwargs)\\n        content = res.read().decode() if not is_stream else b\\'\\'.join(list(res.iter_bytes())).decode()\\n        _write_cache(key, content, \"cachy.jsonl\")\\n        return httpx.Response(status_code=res.status_code, content=content, request=r)', 'output': '', 'msg_type': 'code', 'input_tokens': 264, 'output_tokens': {}}, {'id': '_60a2bd17', 'time_run': '2026-03-31T09:45:05.947106+00:00', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': 'def _apply_patch(cfp, doms):    \\n    @patch\\n    def send(self:httpx._client.Client, r, **kwargs):\\n        is_stream = kwargs.get(\"stream\")\\n        if not _should_cache(r.url, doms): return self._orig_send(r, **kwargs)\\n        key = _key(r, is_stream=False)\\n        if res := _cache(key,cfp): return httpx.Response(status_code=200, content=res, request=r)\\n        res = self._orig_send(r, **kwargs)\\n        content = res.read().decode() if not is_stream else b\\'\\'.join(list(res.iter_bytes())).decode()\\n        _write_cache(key,content,cfp)\\n        return httpx.Response(status_code=res.status_code, content=content, request=r)', 'output': '', 'msg_type': 'code', 'input_tokens': 260, 'output_tokens': {}}, {'id': '_37ad35fa', 'time_run': '2026-03-31T09:45:06.036394+00:00', 'is_exported': 0, 'skipped': 0, 'bookmark': {}, 'i_collapsed': 0, 'o_collapsed': 0, 'heading_collapsed': False, 'i_clamp': False, 'o_clamp': False, 'pinned': 0, 'run': False, 'content': '#| exports\\ndef _apply_async_patch(cfp, doms, hdrs, debug=False):    \\n    @patch\\n    async def send(self:httpx._client.AsyncClient, r, **kwargs):\\n        is_stream = kwargs.get(\"stream\")\\n        if not _should_cache(r.url, doms): return await self._orig_send(r, **kwargs)\\n        key = _key(r, is_stream=False)\\n        if res := _cache(key,cfp):\\n            if debug: print(f\"\ud83d\udfe2 HIT {key}\\\\n{r.content}\")\\n            return httpx.Response(status_code=res.get(\\'status_code\\', 200), content=res[\\'response\\'], hea\u2026</TRUNCATED>"
}
```

</details>

The two exported cells (`_37ad35fa` and `_950030b0`) look correct — both use `res.get('status_code', 200)` on cache hits and pass `res.status_code` to `_write_cache`. `_write_cache` (`_72a0213b`) also stores `status_code` now. ✅

However, I see several **older non-exported cells** (`_d802ec92`, `_30aa118e`, `_056e9c1e`, `_60a2bd17`) that still have `status_code=200` hardcoded. Those appear to be earlier iterations of the function that were kept in the notebook. They aren't exported (no `#| exports`), so they won't affect the built module — but you might want to clean them up to avoid confusion.

<details class='token-usage-details'><summary>$0.4126</summary>

`total=96,281 | in=95,995 | out=286 | cached=59.5% | cache_new=35,901 | $0.4126`

</details>


I meant the example tests above in this dialog, are they properly showing non-200?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the outputs above:

- **Cell `_1ad7307c`** (direct `APIError` construction): `status_code=401` ✅
- **Cell `_b5140f19`** (fresh request): `status_code=401` ✅  
- **Cell `_90778e19`** (`api_error_from_http`): `status_code=401` ✅

All showing 401 correctly now — the cachy fix is working! No more hardcoded 200s.

<details class='token-usage-details'><summary>$0.0791</summary>

`total=49,514 | in=48,967 | out=547 | cached=94.7% | cache_new=2,353 | $0.0791`

</details>


Does it make sense to make `api_error_from_http` a patch on `httpx.HTTPStatusError`?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It would be consistent with the existing `@patch` on `httpx.Response.error()`. You'd go from:

```python
api_error_from_http(exc, provider="anthropic", model="claude-sonnet-4-20250514")
```

to:

```python
exc.api_error(provider="anthropic", model="claude-sonnet-4-20250514")
```

One consideration: `HTTPStatusError` is an exception class, so patching methods onto it is slightly unusual — but fastcore's `@patch` handles it fine, and it keeps the pattern consistent with `resp.error()`. What do you think?

<details class='token-usage-details'><summary>$0.4756</summary>

`total=49,572 | in=49,171 | out=401 | cached=0.0% | cache_new=48,933 | $0.4756`

</details>


### `api_error_from_event`

Build APIError from provider SSE/event-level error payload.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `api_error_from_event(event: Any, *, provider: str, model: str, endpoint: str)`
- Returns: `APIError`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `normalize`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def api_error_from_event(event, *, provider: str = "", model: str = "", endpoint: str = ""):
    "Build APIError from provider SSE/event-level error payload."
    parsed = _parse_sse_error_event(event)
    return APIError(parsed.message, provider=provider, model=model, endpoint=endpoint, error_type=parsed.type, code=parsed.code, raw=parsed.raw)


Ok let's explain this function now

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

This is a thin adapter that bridges SSE error events to the `APIError` type. It does two things:

1. **Parses** the raw SSE event dict via `_parse_sse_error_event(event)` — which returns `(message, error_type, code, raw)` after handling the three different provider shapes (OpenAI flat, Anthropic/Gemini nested)

2. **Wraps** those parsed fields into an `APIError`, stamping on the caller-provided context (`provider`, `model`, `endpoint`)

Note that unlike `api_error_from_http`, this one has no `status_code` or `request_id` — SSE error events don't carry HTTP-level metadata since they arrive inside an already-open stream.

<details class='token-usage-details'><summary>$0.2870</summary>

`total=49,958 | in=49,749 | out=209 | cached=43.4% | cache_new=27,606 | $0.2870`

</details>


Can we test it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! The SSE test events from earlier are already defined. Try this:

```python
for name, evt in [("OpenAI", oai_sse), ("Anthropic", ant_sse), ("Gemini", gem_sse)]:
    err = api_error_from_event(evt, provider=name.lower())
    print(err)
```

<details class='token-usage-details'><summary>$0.0492</summary>

`total=50,129 | in=49,987 | out=142 | cached=98.4% | cache_new=571 | $0.0492`

</details>


In [ ]:
for name, evt in [("OpenAI", oai_sse), ("Anthropic", ant_sse), ("Gemini", gem_sse)]:
    err = api_error_from_event(evt, provider=name.lower())
    print(err)

APIError(message='Rate limit exceeded', provider='openai', code='rate_limit_exceeded', retryable=True)
APIError(message='Overloaded', provider='anthropic', error_type='overloaded_error', code='overloaded_error', retryable=True)
APIError(message='An internal error has occurred', provider='gemini', error_type='INTERNAL', code='INTERNAL', retryable=True)


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

All three work and are correctly classified as `retryable=True`. But notice the Gemini one — `code=500` instead of `code='INTERNAL'`. That's the misalignment flagged earlier: `_parse_sse_error_event` doesn't promote `status`→`code` like `httpx.Response.error()` does. Worth fixing?

<details class='token-usage-details'><summary>$0.0569</summary>

`total=50,613 | in=50,345 | out=268 | cached=97.7% | cache_new=738 | $0.0569`

</details>


Can you explain the issue in more detail i'm not following

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Compare the two Gemini results:

**HTTP error** (via `resp.error()`):
```
code='INVALID_ARGUMENT'   ← semantic string
```

**SSE error** (via `_parse_sse_error_event`):
```
code=500                  ← numeric
```

Both Gemini error payloads have the same structure: `{"code": 500, "status": "INTERNAL", ...}`. The HTTP path has this promotion logic at the end:

```python
if (code is None or isinstance(code, int)) and isinstance(status, str) and status:
    code = status
```

This replaces numeric `code` (500) with the semantic `status` string (`"INTERNAL"`). But `_parse_sse_error_event` doesn't have that — it just takes `code` as-is from the dict, so Gemini SSE errors keep the numeric 500 instead of getting promoted to `"INTERNAL"`.

<details class='token-usage-details'><summary>$0.0521</summary>

`total=50,760 | in=50,497 | out=263 | cached=98.9% | cache_new=337 | $0.0521`

</details>


Sure let's fix it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Add this just before the `return` in `_parse_sse_error_event`:

```python
if (code is None or isinstance(code, int)) and isinstance(et, str) and et: code = et
```

Note: in the SSE function, `et` already holds `e.get("status")` (via the `e.get("type") or e.get("status")` chain), so for Gemini where `type` is absent but `status` is `"INTERNAL"`, `et` will be `"INTERNAL"` — and this line will promote `code` from `500` to `"INTERNAL"`.

<details class='token-usage-details'><summary>$0.0507</summary>

`total=51,017 | in=50,787 | out=230 | cached=99.0% | cache_new=295 | $0.0507`

</details>


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Want to re-run the test to verify the fix?

```python
for name, evt in [("OpenAI", oai_sse), ("Anthropic", ant_sse), ("Gemini", gem_sse)]:
    err = api_error_from_event(evt, provider=name.lower())
    print(err)
```

Gemini should now show `code='INTERNAL'` instead of `code=500`.

<details class='token-usage-details'><summary>$0.2978</summary>

`total=51,157 | in=50,984 | out=173 | cached=42.4% | cache_new=29,199 | $0.2978`

</details>


I just did see above

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the output from cell `_2ac52ab8` — Gemini now shows `code='INTERNAL'` instead of `code=500`. The promotion is working correctly across all three providers. ✅

<details class='token-usage-details'><summary>$0.0494</summary>

`total=51,374 | in=51,150 | out=224 | cached=99.3% | cache_new=141 | $0.0494`

</details>


Ok, are we done with this dialog? Anything else to check/improve/fix?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A few loose ends I noticed:

1. **`_repr_markdown_`** on `APIError` — we discussed it but caught the naming issue (`_repr_markdown_` not `__repr_markdown__`). Did you end up adding it?

2. **Cachy cleanup** — there are several old non-exported cells in the cachy notebook still showing `status_code=200`. Not harmful (they're not exported), but worth cleaning up.

3. **SSE `error` on `SSEvent`** — you mentioned potentially patching `_parse_sse_error_event` onto the `SSEvent` class in the `sse` dialog later.

<details class='token-usage-details'><summary>$0.0599</summary>

`total=51,781 | in=51,276 | out=505 | cached=99.3% | cache_new=110 | $0.0599`

</details>


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()